## Merge Two Point Clouds

In [ ]:
import pdal
import json

def merge_laz_files(input_files, output_filename):
    # Build the pipeline dictionary: read inputs -> merge -> write output
    pipeline_dict = []
    
    # add all input files to the pipeline as readers
    for file in input_files:
        pipeline_dict.append(file)
        
    # add the merge filter
    pipeline_dict.append({
        "type": "filters.merge"
    })
    
    # add the writer to save the merged point cloud
    pipeline_dict.append({
        "type": "writers.las",
        "filename": output_filename
    })
    
    # convert the dictionary to a JSON string for PDAL
    pipeline_json = json.dumps(pipeline_dict)
    
    # create and execute the pipeline
    print(f"Merging {len(input_files)} files into {output_filename}...")
    pipeline = pdal.Pipeline(pipeline_json)
    
    try:
        count = pipeline.execute()
        print(f"Success! Merged point cloud contains {count} points.\n")
    except RuntimeError as e:
        print(f"An error occurred: {e}")


files_2023 = ["GadValley_PtCld_2023_pt1.laz", "GadValley_PtCld_2023_pt2.laz"]

files_2025 = ["GadValley_PtCld_2025_pt1.laz", "GadValley_PtCld_2025_pt2.laz"]

# merge files
merge_laz_files(files_2023, "GadValley_PtCld_2023_merged.laz")

merge_laz_files(files_2025, "GadValley_PtCld_2025_merged.laz")

## Save Master Core Points

In [ ]:
import laspy
las = laspy.read("C:/Users/alamsal/Downloads/lidar/m3c2/GadValley_PtCld_2023_merged_sample.laz")

subsampled_las = laspy.LasData(las.header)
subsampled_las.points = las.points[::100].copy() 

subsampled_las.write("C:/Users/alamsal/Downloads/lidar/m3c2/MASTER_corepoints.laz")
print("Saved Master Core Points!")

## Convert LAZ to Numpy Arrays

In [ ]:
import numpy as np
import laspy
import matplotlib.pyplot as plt
from m3c2 import compute_m3c2  # Importing from your saved m3c2.py file

def load_las_to_numpy(filepath): # reads a las file and extracts X, Y, Z coordinates into an Nn3 array
    las = laspy.read(filepath)
    return np.vstack((las.x, las.y, las.z)).T

# load las files
cloud_ref = load_las_to_numpy("C:/Users/alamsal/Downloads/lidar/m3c2/GadValley_PtCld_2023_merged_sample.laz")
cloud_cmp = load_las_to_numpy("C:/Users/alamsal/Downloads/lidar/m3c2/GadValley_PtCld_2025_merged_sample.laz")

# to optimize the performance we are using every 100th point as the core point
corepoints = cloud_ref[::100] 

print(f"Reference Cloud Points: {cloud_ref.shape[0]}")
print(f"Compared Cloud Points: {cloud_cmp.shape[0]}")
print(f"Core Points for Calculation: {corepoints.shape[0]}")

## Compute M3C2 Distances

In [ ]:
# Run the calculation
results = compute_m3c2(cloud_ref=cloud_ref, cloud_cmp=cloud_cmp, core_points=corepoints, normal_radius=2.0, proj_diameter=1.0, max_depth=5.0, reg_error=0.0707)

# Extract your arrays for plotting
m3c2_distances = results["distances"]
directions = results["normals"]
lodetection = results["lod95"]
change_sign = results["significant"]

## Save M3C2 Output to LAZ

In [ ]:
import os
import laspy
import numpy as np

def save_m3c2_to_laz(filepath, original_header, corepoints, distances, normals, lod95, significant):

    header = laspy.LasHeader(point_format=6, version="1.4")
    
    header.offsets = original_header.offsets
    header.scales = original_header.scales 
    
    header.add_extra_dim(laspy.ExtraBytesParams(name="M3C2_Distance", type=np.float64))
    header.add_extra_dim(laspy.ExtraBytesParams(name="LOD95", type=np.float64))
    header.add_extra_dim(laspy.ExtraBytesParams(name="Significant_Change", type=np.uint8))
    header.add_extra_dim(laspy.ExtraBytesParams(name="Normal_X", type=np.float64))
    header.add_extra_dim(laspy.ExtraBytesParams(name="Normal_Y", type=np.float64))
    header.add_extra_dim(laspy.ExtraBytesParams(name="Normal_Z", type=np.float64))
    
    # create a new LAS file with the header
    las = laspy.LasData(header)
    
    # fill the X, Y, Z coordinates
    las.x = corepoints[:, 0]
    las.y = corepoints[:, 1]
    las.z = corepoints[:, 2]
    
    # fill custom dimensions
    las.M3C2_Distance = distances
    las.LOD95 = lod95
    las.Significant_Change = significant.astype(np.uint8) 
    las.Normal_X = normals[:, 0]
    las.Normal_Y = normals[:, 1]
    las.Normal_Z = normals[:, 2]
    
    # write to laz file
    las.write(filepath)
    print(f"Successfully compressed and saved {len(corepoints)} M3C2 points to {filepath}")

    # grab the header from your original reference cloud
original_las = laspy.read("C:/Users/alamsal/Downloads/lidar/m3c2/GadValley_PtCld_2023_merged_sample.laz")
original_header = original_las.header

# save the results using that header's math
output_path = "C:/Users/alamsal/Downloads/lidar/m3c2/M3C2_results_python.laz"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

save_m3c2_to_laz(filepath=output_path, original_header=original_header, corepoints=corepoints, distances=m3c2_distances, normals=directions, lod95=lodetection, significant=change_sign)

## Visualize M3C2 Results in 3D

In [ ]:
# create the figure
fig, ax = plt.subplots(1, 1, figsize=(14, 14), subplot_kw={"projection": "3d"})

# plot the distances
d = ax.scatter(corepoints[:, 0], corepoints[:, 1], corepoints[:, 2], c=m3c2_distances, cmap="seismic_r", vmin=-5.0, vmax=5.0, s=1,)
plt.colorbar(d, format=("%.2f"), label="Distance [m]", ax=ax, shrink=0.5, pad=0.15)

# add plot elements
ax.set_xlabel("Easting [m]")
ax.set_ylabel("Northing [m]")
ax.set_aspect("equal", adjustable='box') 
ax.view_init(elev=30.0, azim=120.0)

plt.tight_layout()
plt.show()

## Visualize Changes Above Imagery

In [ ]:
import matplotlib.pyplot as plt
import contextily as cx
from matplotlib_scalebar.scalebar import ScaleBar

fig, ax = plt.subplots(1, 1, figsize=(14, 14))

d = ax.scatter(corepoints[:, 0], corepoints[:, 1], c=m3c2_distances, cmap="Spectral", vmin=-5.0, vmax=5.0, s=2)

cbar = plt.colorbar(d, format=("%.2f"), ax=ax, shrink=0.5, pad=0.02)
cbar.set_label("M3C2 Surface Change [m]", fontsize=14, weight='bold')

cx.add_basemap(ax, crs="EPSG:32612", source=cx.providers.Esri.WorldImagery, alpha=0.7)

scalebar = ScaleBar(dx=1, units="m", location="lower right", box_alpha=0.7, scale_formatter=lambda value, unit: f"{value} {unit}")
ax.add_artist(scalebar)

ax.annotate('N', xy=(0.05, 0.95), xytext=(0.05, 0.88), arrowprops=dict(facecolor='black', width=6, headwidth=18), ha='center', va='center', fontsize=22, weight='bold',xycoords=ax.transAxes)

ax.set_xlabel("Easting [m]", fontsize=12, weight='bold')
ax.set_ylabel("Northing [m]", fontsize=12, weight='bold')
ax.set_aspect("equal", adjustable='box') 
ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()